# **Track Network - Bogie Test Jig**

### Data Fetching

In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_19988\3393819530.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [2]:
keywords = ["BogieTestJig"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.head(5)


,filename,workorder_id,json_data
427,TN_PM_MTH_BogieTestJig_4000512660.pdf,4.000513e+09,"{'notification': {'notification_no': 'NA', 'no..."
889,TN_PM_MTH_BogieTestJig_4000454378.pdf,4.000454e+09,"{'notification': {'notification_no': 'NA', 'no..."
895,TN_PM_MTH_BogieTestJig_4000459781.pdf,4.000460e+09,"{'notification': {'notification_no': 'NA', 'no..."
911,TN_PM_MTH_BogieTestJig_4000464327.pdf,4.000464e+09,"{'notification': {'notification_no': 'NA', 'no..."
933,TN_PM_MTH_BogieTestJig_4000479305.pdf,4.000479e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_like(val):
    """Detect NA-like values."""
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    """Return keys in dict where value is NA-like."""
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]


def clean_value(val):
    """Recursively clean NA-like values in dict, list, string."""
    if isinstance(val, str):
        return '' if pattern_na.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

def extract_leaf_keys(d, parent=''):
    """Extract flattened leaf keys from nested dict."""
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

In [10]:
import string, re

def clean_component(name):
    if not isinstance(name, str) or not name.strip():
        return None

    name = name.lower().strip()
    name = re.sub(r"[^\w]+", "_", name)
    return name


def flatten_json(data):
    flat = {}

    bogie = data.get("bogie_test_jig")
    if not isinstance(bogie, dict):
        return flat

    for key, obj in bogie.items():

        if isinstance(obj, dict) and "component" in obj:
            component = clean_component(obj.get("component"))
            if component is None:
                continue

            for k, v in obj.items():
                if k == "component":
                    continue

                clean_k = "completed" if k == "completed?" else k
                flat[f"{component}.{clean_k}"] = v

        elif key == "technician" and isinstance(obj, dict):
            flat["technician.technician_id"] = obj.get("signature")
            flat["technician.date"] = obj.get("date")

        elif key == "guideway" and isinstance(obj, dict):
            flat["supervisor.supervisor_id"] = obj.get("supervisor")
            flat["supervisor.date"] = obj.get("date")

        elif key == "any_additional_work_that_requires_planning":
            flat["any_additional_work_that_requires_planning"] = obj

    return flat

df_bogie = df.copy()

df_bogie['bogie_test_jig'] = df_bogie['json_data'].apply(
    lambda x: x.get('bogie_test_jig') if isinstance(x, dict) else None
)

df_bogie = df_bogie[df_bogie['bogie_test_jig'].notnull()].copy()

flattened_rows = [
    flatten_json(row)
    for row in df_bogie['json_data']
]

bogie_df = pd.DataFrame(flattened_rows)
bogie_df.index = df_bogie.index

# Put identifiers in front
bogie_df.insert(0, 'workorder_id', df_bogie['workorder_id'].astype('Int64'))
bogie_df.insert(1, 'filename', df_bogie['filename'])

front_cols = ['workorder_id', 'filename']

bogie_cols = sorted(
    [c for c in bogie_df.columns if re.match(r'^[a-z]\.', c)]
)

tail_cols = [
    c for c in bogie_df.columns
    if c not in front_cols + bogie_cols
]

bogie_df = bogie_df[front_cols + bogie_cols + tail_cols]

### Output Excel

In [11]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/tnm/bogie_test_jig.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    bogie_df.to_excel(writer, index=False, sheet_name='bogie_test_jig'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/tnm/bogie_test_jig.xlsx' (replaced existing sheet)
